# 🚀 PCB Defect Image Synthesis & Quality Filter Pipeline
**Bài báo tham chiếu**: *"A Synthetic Defect Image Generation Method for Enhanced Industrial Defect Detection Based on Inpainting Diffusion and Context-aware Mask Generation"* (Journal of Intelligent Manufacturing, 2026)

Notebook này được thiết kế để huấn luyện và thực thi toàn bộ pipeline trực tiếp trên **Google Colab (GPU T4 / A100)**:
1. **Step 1**: Context-Aware Mask Generation (DINOv2 backbone + Top-5 Sampling)
2. **Step 2**: LoRA Inpainting Diffusion ($r=8, \alpha=16$, 6-token Textual Inversion, 9-channel UNet, Latent Overwriting)
3. **Step 3**: Confidence-based Quality Filtering bằng YOLO11s (100 epochs AdamW, Target-aware IoU matching, loại 25% thấp nhất)

In [ ]:
# ── 1. Kiểm tra cấu hình GPU ─────────────────────────────
!nvidia-smi

## 📂 Bước 1: Kết nối Google Drive & Giải nén Dữ liệu
Đảm bảo bạn đã tải file `diffusion_colab.zip` lên Google Drive (ví dụ để ở thư mục gốc `MyDrive`).

In [ ]:
from google.colab import drive
import os

# Kết nối Google Drive
drive.mount('/content/drive')

# Đường dẫn tới file ZIP trên Drive và thư mục làm việc trên Colab
ZIP_PATH = '/content/drive/MyDrive/diffusion_colab.zip'
WORK_DIR = '/content/pcb_diffusion'

# Giải nén nội bộ Colab SSD (rất nhanh, chỉ mất ~5-10 giây)
!mkdir -p {WORK_DIR}
!unzip -q {ZIP_PATH} -d {WORK_DIR}

%cd {WORK_DIR}
!ls -la

## 📦 Bước 2: Cài đặt các thư viện phụ thuộc

In [ ]:
%pip install -q diffusers transformers accelerate peft ultralytics opencv-python pyyaml

## 🔧 Bước 3: Đồng bộ hóa đường dẫn môi trường Colab Linux
Lệnh này tự động cập nhật các file `id_setup.yaml`, `npi_setup.yaml`, `prepared_splits/*.txt`, `train_metadata.json` khớp 100% với đường dẫn `/content/pcb_diffusion/data`.

In [ ]:
!python setup_env_paths.py

## ⚡ Bước 4: Chạy Toàn Bộ Pipeline từ A -> Z (One-Click Full Pipeline)
Chạy toàn bộ 3 bước của bài báo bằng GPU với đầy đủ quá trình huấn luyện thực tế:
- **Step 1**: Sinh mặt nạ ngữ cảnh CAMG (Top-5 sampling)
- **Step 2a**: Huấn luyện LoRA ($r=8, \alpha=16$) + Textual Inversion 10 epochs
- **Step 2b**: Sinh ảnh lỗi với Stable Diffusion Inpainting + Latent Overwriting (50 steps)
- **Step 3**: Huấn luyện YOLO11s Judge 100 epochs với AdamW
- **Step 3 Filter**: Lọc độ tin cậy Target-aware IoU, loại 25% thấp nhất và xuất `augmented_dataset.yaml`

In [ ]:
!python run_pipeline.py full \
    --image_dir data/pcb_normal \
    --real_defect_dir data/pcb-defect-dataset \
    --metadata_path data/train_metadata.json \
    --data_yaml data/id_setup.yaml \
    --output_dir outputs/pcb_synthesis \
    --lora_epochs 10 \
    --lora_batch 32 \
    --inpaint_steps 50 \
    --guidance 7.5 \
    --yolo_size s \
    --yolo_epochs 100 \
    --reject_ratio 0.25 \
    --device cuda

## 💾 Bước 5: Sao lưu kết quả và Model Weights về Google Drive
Lưu ảnh accepted, file nhãn YOLO, LoRA checkpoints và best.pt của YOLO11 về Drive vĩnh viễn:

In [ ]:
BACKUP_DIR = '/content/drive/MyDrive/PCB_Project_Outputs'
!mkdir -p {BACKUP_DIR}
!cp -r outputs/pcb_synthesis/* {BACKUP_DIR}/
print(f"✅ Đã sao lưu toàn bộ kết quả về: {BACKUP_DIR}")

## 📊 Bước 6: Trực quan hóa kết quả (Báo cáo & Hiển thị ảnh mẫu)

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

report_path = Path('outputs/pcb_synthesis/pipeline_report.json')
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print("=" * 50)
    print("📈 TỔNG KẾT PIPELINE:")
    print(f"  Tổng mask đã tạo:      {report.get('total_masks')}")
    print(f"  Tổng ảnh đã sinh:      {report.get('total_generated')}")
    print(f"  Tổng ảnh đạt chuẩn:    {report.get('total_accepted')}")
    print(f"  Tỷ lệ loại bỏ:         {report.get('rejected_ratio') * 100:.0f}%")
    print(f"  Thời gian chạy:        {report.get('elapsed_minutes')} phút")
    print(f"  Augmented YAML:        {report.get('augmented_yaml')}")
    print("=" * 50)

# Hiển thị mẫu các ảnh khuyết tật nhân tạo accepted
acc_meta_path = Path('outputs/pcb_synthesis/filtered/accepted_metadata.json')
if acc_meta_path.exists():
    accepted_items = json.loads(acc_meta_path.read_text(encoding='utf-8'))
    samples = accepted_items[:min(4, len(accepted_items))]

    fig, axes = plt.subplots(1, len(samples), figsize=(16, 4))
    if len(samples) == 1:
        axes = [axes]
    for ax, item in zip(axes, samples):
        img_p = item.get('filtered_image_path') or item.get('synthetic_image_path')
        if Path(img_p).exists():
            ax.imshow(Image.open(img_p))
            ax.set_title(f"{item['defect_type']}\nConf: {item.get('confidence', 0):.3f}", fontsize=10)
        ax.axis('off')
    plt.suptitle("Sample Accepted Synthetic Defect Images", fontsize=14, fontweight='bold')
    plt.show()